In [2]:
import torch
import torch.nn as nn
from torch import Tensor

embeding时，我们将embeding分量初始化为均值为0，标准差为 (d_model)^(-1/2)。每一个分量都满足均值为0，标准差为(d_model)^(-1/2)的正太分布，每一个embedding向量的范数是1。
E(||x||^2)=\sum_{i=1}^{d_model}E{(x_i)^2}
E((x_i)^2) = E(((x_i)-E(x))^2) = Var(x), 因为 E(x_i) = 0 和 Var(x_i)= d_model^(-1)，所以E((x_i)^2) = d_model^(-1)。
所以E(||x||^2)=1


In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size: int, d_model: int) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        # 可训练参数：每行是一个 token 的向量
        self.weight = nn.Parameter(torch.empty(vocab_size, d_model))
        nn.init.normal_(self.weight, mean=0.0, std=d_model ** -0.5)

    def forward(self, token_ids: Tensor) -> Tensor:
        """
        token_ids: (B, L), dtype=torch.long
        返回: (B, L, D)
        """
                # (B, L) -> (B, L, D)
        x = self.weight[token_ids]          # 花式索引即查表
        if self.scale:
            x = x * math.sqrt(self.d_model) 
        return x
        
    

In [ ]:
位置编码
我们将每一个token的每两个特征设置为一组，设token是整个序列中的第i个token，该token的第j组。
则有：p_{i,2j} = sin(\frac{i}{10000^{2j/d}})
p_{i,2j+1} = cos(\frac{i}{10000^{2j/d}})
\(i\) 决定当前采样位置，\(j\) 决定使用哪一种频率。固定 \(j\) 沿 \(i\) 观察时频率不变；切换 \(j\) 时，频率才会改变。

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        # 构造分母
        index = torch.arange(0, d_model, 2).float()
        scale = -math.log(10000.0) / d_model
        div_term = torch.exp(index * scale)
        position = torch.arange(max_len).unsqueeze(1)
        pe[:,0::2] = torch.sin(div_term * position)
        pe[:,1::2] = torch.cos(div_term * position)
        self.register_buffer("pe",pe.unsqueeze(0))

    def forward(self, x: Tensor, start_pos: int = 0) -> Tensor:
        L = x.shape(1)
        return x + self.pe[:,start_pos:start_pos+L,:]
        

    
    

多层感知机（MLP）：由多个全连接层堆叠而成，每层与上一层完全连接。核心是非线性激活函数——没有它多层会退化为单层线性变换。常见激活函数（ReLU 等）是固定函数，没有可训练参数；可训练的是全连接层的权重和偏置。

In [ ]:
class PositionWiseFFN(nn.Module):
    def __init__(self, d_model, d_ff) -> None:
        super().__init__()
        self.dense1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.dense2 = nn.Linear(d_ff, d_model)

    def forward(self, x: Tensor) -> Tensor:
        return self.dense2(self.relu(self.dense1(x)))

标准化
减去均值：得到相对“中心位置”的偏移，消除了整体平移。
除以标准差：把偏移量按该组数据自身的波动尺度来衡量；结果表示“离均值多少个标准差”，而不是一般意义上的比例。
标准化通常是
\[
z=\frac{x-\mu}{\sigma}
\]这样处理后，数据均值约为 0、标准差约为 1，方便不同量纲、不同数值范围的特征被模型以更接近的尺度处理。

LayerNorm和batchNorm
layerNorm在同一个token中对所有特征进行标准化。果某个特征在这个 token 里是异常大的 outlier，会把其他特征一起压扁。特征维度少或特征含义差异大时，LN 的效果受影响
batchNorm对同一个Batch中对所有token的同一个特征进行标准化。特征之间互不干扰，保留了每个通道独立的分布形状

In [3]:
ln = nn.LayerNorm(2)
bn = nn.BatchNorm1d(2)
X = torch.tensor([[1, 2], [2, 3]], dtype=torch.float32)
# 在训练模式下计算X的均值和方差
print('layer norm:', ln(X), '\nbatch norm:', bn(X))

layer norm: tensor([[-1.0000,  1.0000],
        [-1.0000,  1.0000]], grad_fn=<NativeLayerNormBackward0>) 
batch norm: tensor([[-1.0000, -1.0000],
        [ 1.0000,  1.0000]], grad_fn=<NativeBatchNormBackward0>)


残差网络
从y = F(x)变成 y = x + F(x) 时，实际上神经网络学习到的是从F(x)如何得到target y,到F(x)如何得到 target y -x 。如果神经网络什么也没学到，也就是 target y -x  = 0，那至少在y = x + F(x) 这个式子中，x还能继续向下传。

In [ ]:
class AddNorm(nn.Module):
    # layernomr需要先初始化才能用
    def __init__(self, d_model, dropout) -> None:
        super().__init__()
        self.ln = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X, Y) -> Tensor:
        return self.ln(X + self.dropout(Y))
